# Tape Preview — Canny + HoughLinesP

Camera has chromatic aberration (pink fringing) and lens shading.  
Avoid color. Detect tape as **line segments** via edge detection → robust to lighting.

Pipeline: ROI crop (bottom) → grayscale → blur → Canny → HoughLinesP → filter by angle → pick innermost line each side → midpoint.

View: **frame + detection** (left) + **edge map** (right).

In [1]:
import time
import numpy as np
import cv2
import ipywidgets as widgets
from IPython.display import display

def gstreamer_pipeline(width=640, height=480, fps=30):
    return (
        f"nvarguscamerasrc ! "
        f"video/x-raw(memory:NVMM), width={width}, height={height}, "
        f"format=NV12, framerate={fps}/1 ! "
        f"nvvidconv flip-method=0 ! "
        f"video/x-raw, width={width}, height={height}, format=BGRx ! "
        f"videoconvert ! video/x-raw, format=BGR ! appsink"
    )

cap = cv2.VideoCapture(gstreamer_pipeline(), cv2.CAP_GSTREAMER)
assert cap.isOpened(), "Camera failed to open!"

image_widget = widgets.Image(format='jpeg', width=1280, height=480)

wide = widgets.Layout(width='500px')
style = {'description_width': '120px'}
canny_lo     = widgets.IntSlider(value=50,  min=0,  max=255, description='canny lo',    layout=wide, style=style, continuous_update=True)
canny_hi     = widgets.IntSlider(value=150, min=0,  max=500, description='canny hi',    layout=wide, style=style, continuous_update=True)
blur         = widgets.IntSlider(value=5,   min=1,  max=21,  step=2, description='blur k',      layout=wide, style=style, continuous_update=True)
min_line_len = widgets.IntSlider(value=40,  min=5,  max=300, description='min line len', layout=wide, style=style, continuous_update=True)
max_line_gap = widgets.IntSlider(value=10,  min=0,  max=100, description='max line gap', layout=wide, style=style, continuous_update=True)
min_angle    = widgets.IntSlider(value=20,  min=0,  max=89,  description='min angle deg', layout=wide, style=style, continuous_update=True)
roi_pct      = widgets.IntSlider(value=40,  min=20, max=100, description='roi % (bottom)', layout=wide, style=style, continuous_update=True)
live = widgets.Label(value='live values: --')

display(image_widget)
display(widgets.VBox([canny_lo, canny_hi, blur, min_line_len, max_line_gap, min_angle, roi_pct, live]))

def bgr_to_jpeg(frame):
    _, buf = cv2.imencode('.jpeg', frame)
    return bytes(buf)

print("Camera ready.")

Image(value=b'', format='jpeg', height='480', width='1280')

Camera ready.


In [2]:
from IPython import get_ipython
kernel = get_ipython().kernel

try:
    while True:
        kernel.do_one_iteration()

        ret, frame = cap.read()
        if not ret:
            time.sleep(0.01)
            continue

        h, w = frame.shape[:2]
        center_x = w // 2
        bot_pt = (center_x, h - 1)

        clo_v  = canny_lo.value
        chi_v  = canny_hi.value
        k_v    = blur.value if blur.value % 2 == 1 else blur.value + 1
        mll_v  = min_line_len.value
        mlg_v  = max_line_gap.value
        ang_v  = min_angle.value
        roi_v  = roi_pct.value
        live.value = (f'live values: canny={clo_v}/{chi_v} blur={k_v} '
                      f'minLen={mll_v} maxGap={mlg_v} minAng={ang_v} roi={roi_v}%')

        roi_h   = max(1, int(h * roi_v / 100))
        roi_y0  = h - roi_h

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (k_v, k_v), 0)
        roi_gray = gray[roi_y0:, :]

        edges = cv2.Canny(roi_gray, clo_v, chi_v)

        lines = cv2.HoughLinesP(
            edges, rho=1, theta=np.pi / 180, threshold=40,
            minLineLength=mll_v, maxLineGap=mlg_v,
        )

        vis = frame.copy()
        cv2.line(vis, (center_x, 0), (center_x, h - 1), (0, 255, 255), 1)
        cv2.line(vis, (0, roi_y0), (w, roi_y0), (255, 255, 255), 1)

        kept = []  # (mx, my, (x1,y1,x2,y2), angle_deg)
        if lines is not None:
            for seg in lines[:, 0, :]:
                x1, y1, x2, y2 = seg
                dx = x2 - x1
                dy = y2 - y1
                angle = abs(np.degrees(np.arctan2(dy, dx)))  # 0..180
                if angle > 90:
                    angle = 180 - angle
                if angle < ang_v:
                    continue
                mx = (x1 + x2) // 2
                my = (y1 + y2) // 2 + roi_y0
                y1f = y1 + roi_y0
                y2f = y2 + roi_y0
                kept.append((mx, my, (x1, y1f, x2, y2f), angle))

        left_lines  = [k for k in kept if k[0] <  center_x]
        right_lines = [k for k in kept if k[0] >= center_x]

        for mx, my, (x1, y1, x2, y2), _ in left_lines:
            cv2.line(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
        for mx, my, (x1, y1, x2, y2), _ in right_lines:
            cv2.line(vis, (x1, y1), (x2, y2), (0, 128, 255), 2)

        left_pick  = max(left_lines,  key=lambda d: d[0]) if left_lines  else None
        right_pick = min(right_lines, key=lambda d: d[0]) if right_lines else None

        if left_pick and right_pick:
            lx, ly = left_pick[0],  left_pick[1]
            rx, ry = right_pick[0], right_pick[1]
            (lx1, ly1, lx2, ly2) = left_pick[2]
            (rx1, ry1, rx2, ry2) = right_pick[2]
            cv2.line(vis, (lx1, ly1), (lx2, ly2), (0, 255, 0), 3)
            cv2.line(vis, (rx1, ry1), (rx2, ry2), (0, 255, 0), 3)
            mx, my = (lx + rx) // 2, (ly + ry) // 2
            err = mx - center_x
            cv2.circle(vis, (mx, my), 10, (0, 255, 0), -1)
            cv2.arrowedLine(vis, bot_pt, (mx, my), (0, 255, 0), 3, tipLength=0.05)
            cv2.putText(vis, f"mid={mx} err={err:+d} L={len(left_lines)} R={len(right_lines)}",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        else:
            missing = []
            if not left_pick:  missing.append('NO LEFT')
            if not right_pick: missing.append('NO RIGHT')
            cv2.putText(vis, ' '.join(missing),
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        edges_full = np.zeros((h, w), dtype=np.uint8)
        edges_full[roi_y0:, :] = edges
        edges_bgr = cv2.cvtColor(edges_full, cv2.COLOR_GRAY2BGR)
        cv2.line(edges_bgr, (0, roi_y0), (w, roi_y0), (255, 255, 255), 1)
        combined = np.hstack([vis, edges_bgr])
        image_widget.value = bgr_to_jpeg(combined)
        time.sleep(0.03)

except KeyboardInterrupt:
    print("Preview stopped.")

: 

: 

: 

In [ ]:
cap.release()
print("Camera released.")